#Análise do Desempenho de Ana Cristina contra Diferentes Seleções

#Objetivo

Este projeto tem como objetivo analisar o desempenho da atleta Ana Cristina, da Seleção Brasileira Feminina de Vôlei, em partidas oficiais disputadas entre 2024 e 2026.

A investigação busca responder:

- Contra quais seleções a atleta apresenta maior média de pontuação?
- O desempenho contra a Itália é superior ao observado contra outros adversários?
- Quais fundamentos contribuem para esse desempenho?
- Qual fundamento possui maior relação com a pontuação total da atleta?

Para isso, foi realizada a coleta automatizada de dados estatísticos disponibilizados pela Volleyball World.

# Bibliotecas Utilizadas

Nesta etapa são importadas as bibliotecas responsáveis pela coleta, manipulação, análise e visualização dos dados.

In [ ]:
import pandas as pd
import numpy as np
import requests
import re

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Fontes de Dados

Os dados foram coletados a partir das páginas oficiais da Volleyball World referentes às participações de Ana Cristina em competições internacionais.

Competições analisadas:

- Volleyball Nations League 2024
- Jogos Olímpicos de Paris 2024
- Volleyball Nations League 2025
- Volleyball Nations League 2026

In [ ]:
urls = {
    "VNL_2024": "https://br.volleyballworld.com/volleyball/competitions/volleyball-nations-league/2024/players/173849",
    "VNL_2025": "https://br.volleyballworld.com/volleyball/competitions/volleyball-nations-league/2025/players/173849",
    "VNL_2026": "https://br.volleyballworld.com/volleyball/competitions/volleyball-nations-league/players/173849",
    "OLIMPIADAS_2024": "https://br.volleyballworld.com/volleyball/competitions/volleyball-olympic-games-paris-2024/players/173849?utm_source=chatgpt.com"
}

# Funções de Coleta e Tratamento

Nesta seção são definidas as funções responsáveis pela extração, limpeza e padronização dos dados obtidos via web scraping.

As etapas incluem:

- Extração das tabelas
- Identificação do adversário
- Conversão de datas
- Padronização dos nomes das seleções
- Criação de variáveis auxiliares

In [ ]:
def carregar_tabelas(url):
    try:
        tabelas = pd.read_html(url)
        print(f"{len(tabelas)} tabelas encontradas")
        return tabelas

    except Exception as e:
        print("Erro:", e)
        return None

In [ ]:
def extrair_partidas(tabelas):

    df = tabelas[0].copy()

    return df

In [ ]:
def renomear_colunas(df):

    df.columns = [
        "Time_A",
        "Time_B",
        "Data",
        "Pontos",
        "Ataque",
        "Bloqueio",
        "Saque"
    ]

    return df

In [ ]:
def identificar_adversario(df):

    adversarios = []

    for _, row in df.iterrows():

        time_a = str(row["Time_A"])
        time_b = str(row["Time_B"])

        if "Brasil" in time_a:
            adversarios.append(time_b)

        else:
            adversarios.append(time_a)

    df["Adversario"] = adversarios

    return df

In [ ]:
def limpar_adversarios(df):

    df["Adversario"] = (
        df["Adversario"]
        .str.replace(r"[A-Z]{3}$", "", regex=True)
        .str.strip()
    )

    return df

In [ ]:
def converter_data(df):

    df["Data"] = pd.to_datetime(
        df["Data"],
        dayfirst=True
    )

    return df

In [ ]:
def adicionar_metadados(df, competicao):

    df["Competicao"] = competicao

    df["Ano"] = (
        df["Data"]
        .dt.year
    )

    return df

In [ ]:
def criar_variavel_italia(df):

    df["Contra_Italia"] = (
        df["Adversario"] == "Itália"
    ).astype(int)

    return df

In [ ]:
def selecionar_colunas(df):

    return df[
        [
            "Data",
            "Ano",
            "Competicao",
            "Adversario",
            "Pontos",
            "Ataque",
            "Bloqueio",
            "Saque",
            "Contra_Italia"
        ]
    ]

# Construção do Dataset

Após a definição das funções, os dados de todas as competições são coletados e consolidados em um único conjunto de dados.

Cada linha representa uma partida disputada por Ana Cristina.

In [ ]:
todos_dfs = []

for competicao, url in urls.items():

    tabelas = carregar_tabelas(url)

    df = extrair_partidas(tabelas)

    df = renomear_colunas(df)

    df = identificar_adversario(df)

    df = limpar_adversarios(df)

    df = converter_data(df)

    df = adicionar_metadados(
        df,
        competicao
    )

    df = criar_variavel_italia(df)

    df = selecionar_colunas(df)

    todos_dfs.append(df)

7 tabelas encontradas
7 tabelas encontradas
7 tabelas encontradas
7 tabelas encontradas


In [ ]:
dataset_final = pd.concat(
    todos_dfs,
    ignore_index=True
)

# Visão Geral dos Dados

Nesta etapa é realizada uma inspeção inicial do conjunto de dados para verificar sua estrutura e composição.

In [ ]:
print(dataset_final.head())

print(dataset_final.shape)

        Data   Ano Competicao Adversario  Pontos  Ataque  Bloqueio  Saque  \
0 2024-06-23  2024   VNL_2024    Polônia      10       6         3      1   
1 2024-06-22  2024   VNL_2024      Japão       6       6         0      0   
2 2024-06-20  2024   VNL_2024   Thailand      14      12         1      1   
3 2024-06-16  2024   VNL_2024    Turquia       9       6         1      2   
4 2024-06-14  2024   VNL_2024   Bulgaria      15      11         2      2   

   Contra_Italia  
0              0  
1              0  
2              0  
3              0  
4              0  
(36, 9)


# Frequência dos Confrontos

Nem todas as seleções foram enfrentadas a mesma quantidade de vezes.

Antes das análises comparativas, é importante verificar quantos confrontos foram registrados contra cada adversário.

In [ ]:
#Contra quais seleções ela mais pontua?
media_pontos = (
    dataset_final
    .groupby("Adversario")["Pontos"]
    .agg(["count","mean","std"])
    .sort_values("mean", ascending=False)
)

print(media_pontos)

                    count       mean        std
Adversario                                     
Belgium                 1  22.000000        NaN
Itália                  3  20.000000   3.464102
Estados Unidos          3  19.000000   5.567764
Korea                   1  18.000000        NaN
Canada                  2  17.000000   5.656854
Turquia                 3  16.333333   9.451631
Dominican Republic      3  16.333333   9.712535
Czechia                 1  16.000000        NaN
Países Baixos           2  14.000000   7.071068
Japão                   3  11.666667   4.932883
Sérvia                  1  11.000000        NaN
Alemanha                2  10.500000  14.849242
Bulgaria                3   9.333333   8.144528
França                  2   9.000000   5.656854
Thailand                2   7.500000   9.192388
Polônia                 3   6.666667   5.773503
Kenya                   1   6.000000        NaN


# Seleção dos Adversários com Maior Representatividade

Para garantir comparabilidade entre os grupos analisados, serão consideradas apenas as seleções com pelo menos três confrontos registrados.

Esse critério reduz o impacto de médias calculadas a partir de poucas observações.

In [ ]:
#Contagem para apenas os países que tiveram 3 jogos de confronto jogados pela Ana contra o Brasil
contagem = dataset_final["Adversario"].value_counts()

paises_validos = contagem[
    contagem >= 3
].index

df_analise = dataset_final[
    dataset_final["Adversario"].isin(paises_validos)
]

print(df_analise["Adversario"].value_counts())

Adversario
Polônia               3
Japão                 3
Turquia               3
Bulgaria              3
Itália                3
Estados Unidos        3
Dominican Republic    3
Name: count, dtype: int64


# Desempenho Médio por Adversário

Esta análise busca identificar contra quais seleções Ana Cristina apresenta maior produção ofensiva.

Serão calculadas estatísticas descritivas como:

- Média de pontos
- Desvio padrão
- Quantidade de jogos
- Maior pontuação registrada

In [ ]:
media_df = (
    df_analise
    .groupby("Adversario")["Pontos"]
    .agg(
        Jogos="count",
        Media="mean",
        Mediana= "median",
        Desvio="std",
        Maximo="max"
    )
    .sort_values("Media", ascending=False)
)

print(media_df)

                    Jogos      Media  Mediana    Desvio  Maximo
Adversario                                                     
Itália                  3  20.000000     22.0  3.464102      22
Estados Unidos          3  19.000000     20.0  5.567764      24
Dominican Republic      3  16.333333     14.0  9.712535      27
Turquia                 3  16.333333     13.0  9.451631      27
Japão                   3  11.666667     14.0  4.932883      15
Bulgaria                3   9.333333     13.0  8.144528      15
Polônia                 3   6.666667     10.0  5.773503      10


# Média de Pontos por Adversário

O gráfico a seguir apresenta a média de pontos obtida por Ana Cristina contra cada adversário selecionado.

In [ ]:
#plotagem do gráfico
import plotly.express as px

media_df = (
    df_analise
    .groupby("Adversario")["Pontos"]
    .mean()
    .reset_index()
)
fig = px.bar(
    media_df,
    y="Adversario",
    x="Pontos",
    color="Pontos",
    orientation="h",
    title="Média de Pontos por Adversário"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"}
)

fig.show()


# Distribuição das Pontuações

A média isoladamente não é suficiente para compreender o desempenho da atleta.

O boxplot permite avaliar a dispersão dos resultados, identificando adversários nos quais o desempenho foi mais consistente ou mais variável.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

fig = px.box(
    df_analise,
    x="Adversario",
    y="Pontos",
    color="Adversario",
    points="all",
    title="Distribuição dos Pontos por Adversário"
)

medias = (
    df_analise
    .groupby("Adversario")["Pontos"]
    .mean()
    .reset_index()
)

fig.add_trace(
    go.Scatter(
        x=medias["Adversario"],
        y=medias["Pontos"],
        mode="markers",
        marker=dict(
            size=12,
            symbol="diamond",
            color="black"
        ),
        name="Média"
    )
)

fig.show()

# Melhores Atuações

Além das médias por adversário, também é interessante observar os confrontos individuais.

Esta análise identifica as partidas com maior  pontuação da atleta ao longo do período analisado.

Interpretação:

Os maiores picos de pontuação de Ana Cristina ocorreram contra República Dominicana e Turquia, ambos com 27 pontos.

Entretanto, a análise anterior mostrou que esses resultados foram influenciados por atuações específicas.

A Itália, embora não tenha apresentado a maior pontuação individual, aparece entre os melhores desempenhos da atleta e se destaca pela regularidade observada ao longo dos confrontos analisados.

Esse resultado sugere que o diferencial dos jogos contra a Itália não está em um único pico de desempenho, mas na consistência de pontuações elevadas.

In [ ]:
dataset_final["Jogo"] = (
    dataset_final["Adversario"]
    + " - "
    + dataset_final["Competicao"]
)

In [ ]:
top5 = (
    dataset_final
    .sort_values("Pontos", ascending=False)
    [["Jogo","Pontos"]]
    .head(5)
)

In [ ]:
fig = px.bar(
    top5,
    x="Pontos",
    y="Jogo",
    orientation="h",
    text="Pontos",
    title="Top 5 Maiores Pontuações"
)

fig.show()

# Efeito Itália

A Itália apresentou a maior média de pontuação da Ana Cristina entre os adversários analisados, foi realizada uma investigação específica dos confrontos contra esta seleção.

O objetivo é compreender quais fundamentos contribuíram para esse desempenho.

In [ ]:
#Composição dos pontos
#Contra a Itália, os pontos vêm principalmente do ataque ou há contribuição relevante de bloqueio e saque?

italia = df_analise[df_analise["Adversario"] == "Itália"]

italia[["Ataque","Bloqueio","Saque"]].sum()

,0
Ataque,51
Bloqueio,4
Saque,5


Composição dos Pontos Contra a Itália

Os pontos obtidos por Ana Cristina podem ser originados de diferentes fundamentos:

- Ataque
- Bloqueio
- Saque

Esta análise busca identificar a contribuição relativa de cada fundamento para a pontuação total da atleta nos confrontos contra a Itália.

In [ ]:
corr = dataset_final[
    ["Pontos","Ataque","Bloqueio","Saque"]
].corr()

print(corr)

            Pontos    Ataque  Bloqueio     Saque
Pontos    1.000000  0.986921  0.521355  0.464548
Ataque    0.986921  1.000000  0.414735  0.367566
Bloqueio  0.521355  0.414735  1.000000  0.190340
Saque     0.464548  0.367566  0.190340  1.000000


In [ ]:
fundamentos = italia[["Ataque","Bloqueio","Saque"]].sum()

participacao = (
    fundamentos /
    fundamentos.sum()
) * 100

print(participacao)

Ataque      85.000000
Bloqueio     6.666667
Saque        8.333333
dtype: float64


In [ ]:
fundamentos = (
    italia[
        ["Ataque","Bloqueio","Saque"]
    ]
    .mean()
    .reset_index()
)

fundamentos.columns = [
    "Fundamento",
    "Media"
]

Distribuição dos Fundamentos

O gráfico abaixo apresenta a participação percentual de cada fundamento na pontuação obtida contra a Itália.

In [ ]:
fig = px.pie(
    fundamentos,
    names="Fundamento",
    values="Media",
    title="Distribuição dos Pontos Contra a Itália"
)

fig.show()

Pontuação por Confronto Contra a Itália

Além da média geral, é importante observar individualmente cada confronto contra a Itália para verificar se o desempenho foi consistente ao longo do tempo.

In [ ]:
fig = px.bar(
    italia,
    x="Data",
    y="Pontos",
    color="Competicao",
    text="Pontos",
    title="Pontuação de Ana Cristina contra a Itália"
)

fig.show()

# Conclusões

A análise permitiu identificar diferenças no desempenho de Ana Cristina em função do adversário enfrentado.

Entre as seleções analisadas, a Itália apresentou a maior média de pontuação da atleta.

O boxplot mostrou que esse desempenho não foi resultado de uma única partida excepcional, mas sim de confrontos consistentemente produtivos.

A análise dos fundamentos revelou que a maior parte dos pontos obtidos contra a Itália foi proveniente de ações ofensivas de ataque.

Por fim, a matriz de correlação indicou que o ataque possui a associação mais forte com a pontuação total da atleta, reforçando sua importância no desempenho ofensivo observado ao longo das competições analisadas.